# Bronze layer
Load the raw transactions of `online_retail_II.xlsx` (both sheets) as they are into a Delta table.

In [0]:
%pip install openpyxl

In [0]:
import pandas as pd

RAW_FILE = "/Volumes/workspace/retail/raw/online_retail_II.xlsx"

# Read both sheets as text: Invoice and StockCode mix numbers and letters,
# so types are assigned in silver, not guessed here
sheets = pd.read_excel(RAW_FILE, sheet_name=None, dtype=str)

for name, df in sheets.items():
    print(f"{name}: {len(df):,} rows")

In [0]:
# Delta does not allow spaces in column names: use snake_case (the only change made in bronze)
COLUMN_NAMES = {
    "Invoice": "invoice",
    "StockCode": "stock_code",
    "Description": "description",
    "Quantity": "quantity",
    "InvoiceDate": "invoice_date",
    "Price": "price",
    "Customer ID": "customer_id",
    "Country": "country",
}

# Stack the sheets, keeping the sheet each row comes from
pdf = pd.concat(
    [df.assign(_source_sheet=name) for name, df in sheets.items()],
    ignore_index=True,
).rename(columns=COLUMN_NAMES)

# Spark expects None for missing values, not pandas NaN
pdf = pdf.astype(object).where(pdf.notna(), None)

print(f"Total rows: {len(pdf):,}")

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType

# Every column is text in bronze
schema = StructType([StructField(c, StringType(), True) for c in pdf.columns])

df_bronze = (
    spark.createDataFrame(pdf, schema=schema)
    .withColumn("_ingested_at", F.current_timestamp())
)

display(df_bronze.groupBy("_source_sheet").count().orderBy("_source_sheet"))

In [0]:
# Overwrite so the notebook can be re-run from scratch at any time
(
    df_bronze.write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("workspace.retail.bronze_transactions")
)

In [0]:
%sql
SELECT _source_sheet, COUNT(*) AS n_rows, MIN(invoice_date) AS first_date, MAX(invoice_date) AS last_date
FROM workspace.retail.bronze_transactions
GROUP BY _source_sheet
ORDER BY _source_sheet

In [0]:
%sql
COMMENT ON TABLE workspace.retail.bronze_transactions IS
  'Raw UCI Online Retail II transactions (Dec 2009 - Dec 2011), loaded as-is from both sheets of online_retail_II.xlsx. All columns are text. The two sheets overlap between 2010-12-01 and 2010-12-09, so those rows appear twice.';

In [0]:
%sql
-- Missing values must be real nulls, not the text 'nan' (expected: 243,007 nulls and 0 'nan')
SELECT
  COUNT_IF(customer_id IS NULL)  AS customer_id_null,
  COUNT_IF(customer_id = 'nan')  AS customer_id_text_nan,
  COUNT_IF(description IS NULL)  AS description_null,
  COUNT_IF(description = 'nan')  AS description_text_nan
FROM workspace.retail.bronze_transactions